# 18.12 MoE 教師能教 Dense 學生嗎？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**老師MoE、學生Dense，shape不能直接相同嗎？

教師內部可以有許多廚師，學生只留一套Dense廚房；只要輸出詞表與prefix一致，KL不要求內部層數相同。

**把直覺寫成數學：**interface-matched而非architecture-matched；總參數與active成本分別報。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

teacher = TinyLM(ModelConfig(width=16, experts=3, top_k=2))
student = TinyLM(ModelConfig(width=8))
ids = torch.tensor([[1, 2]])
print(teacher(ids)["logits"].shape, student(ids)["logits"].shape)
assert teacher(ids)["logits"].shape == student(ids)["logits"].shape

**如何讀結果：**shape檢查不要求加載隨機MoE作真實teacher；正式用已驗證有能力的checkpoint。

**只改一件事：**只改學生vocab_size，看到白盒接口爲什麼壞了。
